How to build a basic LLM chain using LangChain Expression Language (LCEL):

ChatPromptTemplate: Used to construct structural templates for system and user messages.

ChatOpenAI: Instantiates the Language Model wrapper (specifying a custom model name "gpt-4o-mine" and temperature).

StrOutputParser: Extracts and converts the raw message output from the model into a standard string.

The Pipe Operator (|): LCEL leverages the python __or__ operator to seamlessly chain runnables together (prompt | model | parser), passing the output of one component as the input to the next.


LangChain core concepts and Runnables

In [6]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

True

In [9]:
def demo_basic_chain():
    """Demonstrates a basic chain using LCEL and Runnables."""
    # Component 1: Define the prompt template using LCEL

    #from_template Create a chat prompt template from a template string. Creates a chat template consisting of a single message assumed to be from the human.

    prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer in one sentence: {question}")

    model = ChatOpenAI (model="gpt-40-min", temperature=0.7) # temperature parameter that controls how random or creative the model's responses are.

    parser=StrOutputParser()

    # Compose with pipe operator
    chain = prompt | model | parser

    #Execute the chain with an input
    result=chain.invoke({"question":"What is LangChain"})
    print(f"Response: {result}")

    return chain

if "__name__"=="__main__":
    demo_basic_chain()




In [ ]:
# you write a template with placeholders

# Filling the template
# LangChain provides the .invoke() method on the prompt template.

# Sending it to the LLM

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple words."
)

llm = ChatOpenAI(model="gpt-4.1-mini")

user_input = "Python"

formatted_prompt = prompt.invoke({
    "topic": user_input
})

response = llm.invoke(formatted_prompt)

print(response.content)

ChatPromptTemplate.from_messages() creates a conversation instead of a single text prompt. Instead of making one long string, it creates separate chat messages, so that the LLM receives these as structured chat messages.


Pydantic is a Python library that helps you:

Define the structure of your data.
Validate that the data is correct.
Automatically convert compatible data types when possible.

BaseModel is the class you inherit from to define your own data model (schema).you didn't write an __init__() method. BaseModel creates it for you.

Field lets you add extra information about a field. description documents the field.


In [ ]:

from pydantic import BaseModel, Field

class Student(BaseModel):
    name: str = Field(description="Student's full name")
    age: int = Field(gt=0)

PydanticOutputParser, It uses a Pydantic BaseModel to define the expected output and then parses the model's response into a Python object.


The parser can generate formatting instructions.
It produces instructions similar to:

Return a JSON object with:

language: string

creator: string

year: integer

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

from pydantic import BaseModel, Field

class ProgrammingLanguage(BaseModel):
    language: str = Field(description="Programming language name")
    creator: str = Field(description="Creator's name")
    year: int = Field(description="Year it was first released")

parser = PydanticOutputParser(
    pydantic_object=ProgrammingLanguage #ProgrammingLanguage is your schema.PydanticOutputParser will try to convert the LLM's output into that schema.
)

format_instructions = parser.get_format_instructions()

print(format_instructions)

#Now result is not a dictionary.It is a Pydantic object.

#OLD WAY

In [ ]:
class Person (BaseModel):
    name:str = Field(description="The person's name")
    age: int= Field(description="The person's age")
    ocupation: str = Field(description="The person's occupation")
    parser = PydanticOutputParser (pydantic_object=Person)

prompt = ChatPromptTemplate.from_template(
"Return a JSON object with 'name', 'age', and 'occupation' for: {description}"
).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

result = chain.invoke({"description": "A 30-year-old artist named Maria"})

print(result)



Old way (using PydanticOutputParser)

You had to:

Create a BaseModel

Create a PydanticOutputParser

Add format_instructions to the prompt

Ask the model to return JSON

Parse the JSON

In [ ]:
parser = PydanticOutputParser(
    pydantic_object=LanguageInfo
)

prompt = PromptTemplate(
    template="""
    {format_instructions}

    {question}
    """
)

New way (with_structured_output())

Modern chat models can produce structured outputs directly.
No parser, no format instructions, no manual parsing needed.

In [ ]:
from pydantic import BaseModel

class LanguageInfo(BaseModel):
    language: str
    creator: str
    year: int
    
structured_llm = llm.with_structured_output(LanguageInfo)

In [ ]:
#Structured Output
class MovieReview (BaseModel):

    title: str = Field(description="The title of the movie")
    review: str = Field(description="A brief review of the movie")
    rating: int = Field(description="The rating of the movie out of 10")

#Bind the schema to the model
structured_model = llm.with_structured_output(MovieReview)

result=structured_model.invoke("Review:Inception is a mind-blowing thriller.9/10")
print(result)

In [ ]:
#Output Parsers and Structured Output in LangChain V.1


# List and Optional(This field may contain a value of the given type, or it may be None) come from Python's typing module. They are type hints, which tell Python (and libraries like Pydantic and LangChain) what type of data is expected.

from langchain_core.output_parsers import (
StrOutputParser,
JsonOutputParser,
PydanticOutputParser,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List, Optional
from dotenv import load_dotenv


default_factory is a feature in Pydantic's Field() that creates a new default value every time a model object is created.

This is especially important for mutable objects like:

lists []

dictionaries {}

sets set()

timestamps (datetime.now())

In [ ]:
from pydantic import Field

skills: list[str] = Field(default_factory=list)

# Now every time a Student is created, Pydantic effectively does:

skills = list()

# which creates a brand-new list.

Main Runnable Components

These are the ones you should know.

1. RunnableSequence (Most Common)

This is created automatically when you use |.

2. RunnableLambda

This wraps a normal Python function as a Runnable.

Example:

from langchain_core.runnables import RunnableLambda

def double(x):
    return x * 2

runnable = RunnableLambda(double)

print(runnable.invoke(5))

3. RunnablePassthrough

This simply returns the input unchanged.

from langchain_core.runnables import RunnablePassthrough

RunnablePassthrough().invoke("Hello")

Output:

Hello

At first it seems useless.

Its real power is when building dictionaries.

Example:

chain = {
    "question": RunnablePassthrough(), #keeps the input as it is
    "length": RunnableLambda(len) #keeps the computed value
}


4. RunnableParallel

Runs multiple Runnables at the same time on the same input.
RunnableParallel executes the independent runnables in parallel, not one after another.

Example:

from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel(
    uppercase=RunnableLambda(str.upper),
    length=RunnableLambda(len)
)


5. RunnableBranch

Works like an if-else.

In LangChain:

          Input
             │
             ▼
      Condition?
   True        False
   
Chain A      Chain B

This is useful for routing requests based on conditions.

6. RunnableAssign

Adds new values to an existing dictionary without removing the original data.enrich the data instead of replacing it.

7. RunnableMap

Applies the same Runnable to every item in a collection.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

def demo_passthrough_chain():
    # similuatee a retrieve operation
    def fake_retriever(input_dict):
     return " LangChain was created by Harrison Chase in 2022."

    chain = (
    RunnableParallel(context=RunnableLambda(fake_retriever), question=RunnablePassthrough())
    | RunnableLambda (
    lambda x: {"context": x ["context"],
    "question": x["question"] ["question"]})
    | prompt
    | model
    | StrOutputParser()
    )

In [ ]:
from langchain_core.runnables import branch

from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch
#TWO LLM CALLS FOR EACH BRANCH(Whichever is chosen)
def demo_chain_branching():
    """A chain that demonstrates branching functionality"""

    code_prompt=ChatPromptTemplate.from_template("You are a backend expert. Answer : {input} in detail")

    general_prompt=ChatPromptTemplate.from_template("you are a helpful assistant. Answer: {input}")
    
    classifier_prompt=ChatPromptTemplate.from_template("Classify this as 'code' or 'general': {input}\n Return only the classification")

    classifer = classifier_prompt | model | StrOutputParser()

    # Branching chain based on classification
    def is_code_question(input_dict):

        classification = classifer.invoke(input_dict)
        return "code" in classification.lower()

    branch = RunnableBranch (
        (is_code_question, code_prompt | model | StrOutputParser()), #This is a tuple containing two things (condition, runnable). If is_code_question(input_dict) returns True, execute the runnable code chain.
        general_prompt | model | StrOutputParser(), #default branch
        )
    # Test
questions = [
"How do I write a for loop in Python?",
"What's the weather like today?",]

for q in questions:
    result = branch.invoke({"input": q})
    print(f"Q: {q}")
    print(f"A: {result[:100]}...\n")

the syntax is:

RunnableBranch(
    (condition1, runnable1),
    (condition2, runnable2),
    default_runnable
)

from_template() = one message template
from_messages() = multiple messages with different roles


#Looking at the input/output schema of your LangChain chain.


print("Chain input schema:", chain.input_schema.model_json_schema)

Pydantic can convert that schema into JSON Schema
input_schema is a Pydantic model describing the expected input.

Why .model_json_schema?

Pydantic can convert that schema into JSON Schema

#Giving the chain a name for tracing/debugging when it runs.

with_config() lets you attach configuration/metadata to a Runnable without changing what the chain actually does.
When this chain runs, identify this run as greeting_chain.
This is particularly useful for tracing and debugging, especially with LangSmith.

with_config() doesn't execute the chain, it only creates a configured version of the chain.

In [ ]:
print("Chain input schema:", chain.input_schema.model_json_schema)
print("Chain output schema:", chain.output_schema.model_json_schema)

print("Chain input schema:", chain.input_schema.model_json_schema())
print("Chain output schema:", chain.output_schema.model_json_schema())

 # Method 2: Use with_config for tracing
result = chain.with_config( run_name="greeting_chain", ).invoke({"name": "Alice"}) 
print(f"Greeting: {result}")

In [ ]:
#Method 3: Inspect intermediate steps
#Using RunnableLambda for logging

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Say hello to {name}"
)


def log_step(x, step_name=""):
    print(f"[{step_name}] {type(x).__name__}: {str(x)[:100]}")
    return x

debug_chain = (
    prompt
    | RunnableLambda(lambda x: log_step(x, "after_prompt"))
    | model
    | RunnableLambda(lambda x: log_step(x, "after_model"))
    | StrOutputParser()
)

print("\nDebug chain execution:")
result = debug_chain.invoke({"name": "Debug"})
print(f"Greeting: {result}")


ChatPromptTemplate doesn't output a plain string, instead it produces a ChatPromptValue object.

ChatPromptValue
    └── messages
          └── HumanMessage
                 └── content = "Say hello to Alice"


#And after the step "after_model" 
[after_model] AIMessage: content='Hello Alice!'
